# Lincoln Institute — Neighbor State Comparison

Parses Appendix Table 2a ("Homestead Property Taxes for Largest City in Each
State: Median-Valued Homes") from the 50-State Property Tax Comparison Study,
for Kansas (Wichita), Missouri (Kansas City), Nebraska (Omaha), Oklahoma
(Oklahoma City), and Colorado (Denver). Years: 2023, 2024, 2025.

Goal: is Kansas's property tax burden notably different from neighboring
states, and would moving actually reduce it?

In [2]:
import pdfplumber 
import pandas as pd
import re 
import sqlite3

In [3]:
with pdfplumber.open('../data/raw/lincoln_institute/50_state_prop_tax_comparison_for_2025.pdf') as pdf:
    page_62 = pdf.pages[61].extract_text()
    page_63 = pdf.pages[62].extract_text()

combined_text = page_62 + "\n" + page_63
print(combined_text[:2000])

Tax Rate Property Tax Reliance Median Home Value Local Gov’t Spending Classification Ratio*
Rank Tax Rank Impact on Rank Impact on Rank Impact on Rank Impact on
State City (1–75) Rate (1–75) Tax Rate (1–75) Tax Rate (1–75) Tax Rate (1–75) Tax Rate
Florida Jacksonville 35 1.68 29 0.04 47 0.15 39 -0.08 12 0.22
Florida Miami 33 1.79 26 0.07 15 -0.49 32 0.03 7 0.47
Florida Tampa 30 1.87 33 0.01 21 -0.29 21 0.10 10 0.28
Georgia Atlanta 37 1.62 15 0.17 23 -0.22 31 0.03 30 0.01
Hawaii Honolulu** 69 0.88 14 0.17 6 -0.93 75 -0.38 6 0.48
Idaho Boise 72 0.75 11 0.25 20 -0.30 71 -0.29 39 -0.08
Illinois Aurora 12 2.46 4 0.49 54 0.27 50 -0.15 51 -0.17
Illinois Chicago 2 3.36 38 -0.05 41 0.10 11 0.26 11 0.25
Indiana Indianapolis 6 2.72 56 -0.17 65 0.46 28 0.05 13 0.19
Iowa Des Moines 7 2.69 24 0.10 67 0.59 37 -0.06 22 0.06
Kansas Wichita 11 2.56 21 0.11 68 0.59 73 -0.32 14 0.15
Kentucky Louisville 53 1.22 55 -0.17 64 0.46 45 -0.12 65 -0.19
Louisiana New Orleans 31 1.84 54 -0.17 50 0.22 46 -0.13 19 0.

In [4]:
with pdfplumber.open("../data/raw/lincoln_institute/50_state_prop_tax_comparison_for_2025.pdf") as pdf:
    full_text = ""
    for page in pdf.pages:
        page_text = page.extract_text()
        if page_text:
            full_text += page_text + "\n"

# Find where Table 2a actually starts and ends
start_marker = "Appendix Table 2a"
end_marker = "Appendix Table 2b"

start_idx = full_text.find(start_marker)
end_idx = full_text.find(end_marker)

print(f"Table 2a found at character {start_idx}, ends at {end_idx}")

table_2a_text = full_text[start_idx:end_idx]
print(table_2a_text[:2000])

Table 2a found at character 12960, ends at 53820
Appendix Table 2a (page 62).
The average effective tax rate for these 53 cities fell very slightly between 2024 and 2025, from
1.222 percent to 1.213 percent. From 2024 to 2025, however, more cities had increases (27) than
decreases (25), while one city had no change. Billings (MT) experienced the most substantial
change, with an effective tax rate decrease of 37 percent on a median-valued home. Montana
created a graduated property tax structure in 2025 with three tax brackets, which slashed effective
tax rates on homes worth $400,000 or less, with smaller decreases on homes worth up to $1.5
million, and increases on the most valuable homes.
1 The largest cities in each state include 53 cities, because they include Washington, DC, plus two cities in Illinois and
New York since property taxes in Chicago and New York City are so different from those in the rest of each state.
2
Note that differences in property values among cities mean tha

In [5]:
import re

# Find every occurrence of the table title
matches = [m.start() for m in re.finditer(r"Appendix Table 2a", full_text)]
print(f"Found {len(matches)} occurrences at positions: {matches}")

# Print a short preview after each one, so we can see which is the real table
for pos in matches:
    print(f"\n--- Occurrence at {pos} ---")
    print(full_text[pos:pos+150])

Found 3 occurrences at positions: [12960, 146648, 168864]

--- Occurrence at 12960 ---
Appendix Table 2a (page 62).
The average effective tax rate for these 53 cities fell very slightly between 2024 and 2025, from
1.222 percent to 1.213 

--- Occurrence at 146648 ---
Appendix Table 2a) to ensure similar assessment limitation treatment for
properties in the same property tax systems.
However, in six property tax sys

--- Occurrence at 168864 ---
Appendix Table 2a: Homestead Property Taxes for Largest City in Each State: Median-Valued Homes
Tax Rate (%) Tax Bill ($)
Median
Change Change
State C


In [6]:
start_idx = 168864
end_idx = full_text.find("Appendix Table 2b", start_idx)

print(f"Table 2a: {start_idx} to {end_idx}")

table_2a_text = full_text[start_idx:end_idx]
print(table_2a_text)

Table 2a: 168864 to 172406
Appendix Table 2a: Homestead Property Taxes for Largest City in Each State: Median-Valued Homes
Tax Rate (%) Tax Bill ($)
Median
Change Change
State City Rate Rank Amount Rank Home Value
from ’24 from ’24
Alabama Huntsville 0.549% 47 1 ↑ 1,862 49 2 ↑ 339,400
Alaska Anchorage 1.114% 30 2 ↑ 4,785 18 1 ↑ 429,600
Arizona Phoenix 1.044% 33 1 ↑ 4,751 19 1 ↓ 454,900
Arkansas Little Rock 1.123% 28 2 ↑ 3,040 37 1 ↑ 270,700
California Los Angeles 1.179% 24 3 ↑ 11,172 2 - 947,900
Colorado Denver 0.485% 51 2 ↓ 3,085 36 6 ↓ 636,400
Connecticut Bridgeport 1.721% 9 2 ↓ 5,374 11 2 ↓ 312,200
DC Washington 0.708% 41 3 ↑ 5,196 14 2 ↓ 733,400
Delaware Wilmington 1.070% 32 4 ↓ 2,622 44 4 ↓ 245,100
Florida Jacksonville 1.503% 13 - 4,891 16 - 325,300
Georgia Atlanta 0.937% 34 5 ↑ 4,333 21 2 ↑ 462,200
Hawaii Honolulu 0.304% 53 - 2,796 42 1 ↓ 920,600
Idaho Boise 0.637% 45 1 ↑ 3,176 32 - 498,500
Illinois Aurora* 2.735% 2 - 7,906 6 - 289,100
Illinois Chicago 1.548% 12 1 ↓ 5,282 12 2 ↑ 

In [7]:
target_states = ["Kansas", "Missouri", "Nebraska", "Oklahoma", "Colorado"]

records = []

for line in table_2a_text.split("\n"):
    line = line.strip()

    # Only process lines starting with one of our target states
    matched_state = None
    for state in target_states:
        if line.startswith(state):
            matched_state = state
            break

    if not matched_state:
        continue

    # Strip the state name off the front, keep the rest
    rest = line[len(matched_state):].strip()

    # Pattern: City Rate% Rank Change Amount Rank Change HomeValue
    # Change can be "N↑", "N↓", or "-"
    pattern = r'^(.+?)\s+([\d.]+)%\s+(\d+)\s+(\d+\s*[↑↓]|-)\s+([\d,]+)\s+(\d+)\s+(\d+\s*[↑↓]|-)\s+([\d,]+)$'
    match = re.match(pattern, rest)

    if not match:
        print(f"NO MATCH for {matched_state}: {rest}")
        continue

    city, rate, rate_rank, rate_change, amount, amount_rank, amount_change, home_value = match.groups()

    records.append({
        'state': matched_state,
        'city': city.strip().rstrip('*'),  # strip footnote asterisks like Aurora*
        'tax_rate_pct': float(rate),
        'tax_rate_rank': int(rate_rank),
        'tax_bill': float(amount.replace(',', '')),
        'tax_bill_rank': int(amount_rank),
        'median_home_value': float(home_value.replace(',', ''))
    })

neighbor_2025_df = pd.DataFrame(records)
print(f"Rows parsed: {len(neighbor_2025_df)}")
neighbor_2025_df

Rows parsed: 5


,state,city,tax_rate_pct,tax_rate_rank,tax_bill,tax_bill_rank,median_home_value
0,Colorado,Denver,0.485,51,3085.0,36,636400.0
1,Kansas,Wichita,1.118,29,2381.0,46,212900.0
2,Missouri,Kansas City,1.594,11,4169.0,23,261600.0
3,Nebraska,Omaha,1.802,8,4857.0,17,269500.0
4,Oklahoma,Oklahoma City,1.305,19,3383.0,30,259300.0


In [8]:
pdf_files = {
    2023: "../data/raw/lincoln_institute/50_state_property_tax_comparison_for_2023.pdf",
    2024: "../data/raw/lincoln_institute/50_state_property_tax_comparison_for_2024.pdf",
    2025: "../data/raw/lincoln_institute/50_state_prop_tax_comparison_for_2025.pdf",
}

all_years_records = []

for year, filepath in pdf_files.items():
    with pdfplumber.open(filepath) as pdf:
        full_text = ""
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                full_text += page_text + "\n"

    # Find the REAL Table 2a occurrence (skip narrative references)
    matches = [m.start() for m in re.finditer(r"Appendix Table 2a", full_text)]

    real_start = None
    for pos in matches:
        snippet = full_text[pos:pos+100]
        if "Tax Rate (%)" in full_text[pos:pos+300]:
            real_start = pos
            break

    if real_start is None:
        print(f"WARNING: couldn't find real Table 2a for {year}")
        continue

    end_idx = full_text.find("Appendix Table 2b", real_start)
    table_text = full_text[real_start:end_idx]

    for line in table_text.split("\n"):
        line = line.strip()
        matched_state = None
        for state in target_states:
            if line.startswith(state):
                matched_state = state
                break
        if not matched_state:
            continue

        rest = line[len(matched_state):].strip()
        pattern = r'^(.+?)\s+([\d.]+)%\s+(\d+)\s+(\d+\s*[↑↓]|-)\s+([\d,]+)\s+(\d+)\s+(\d+\s*[↑↓]|-)\s+([\d,]+)$'
        match = re.match(pattern, rest)

        if not match:
            print(f"NO MATCH ({year}, {matched_state}): {rest}")
            continue

        city, rate, rate_rank, rate_change, amount, amount_rank, amount_change, home_value = match.groups()

        all_years_records.append({
            'year': year,
            'state': matched_state,
            'city': city.strip().rstrip('*'),
            'tax_rate_pct': float(rate),
            'tax_rate_rank': int(rate_rank),
            'tax_bill': float(amount.replace(',', '')),
            'tax_bill_rank': int(amount_rank),
            'median_home_value': float(home_value.replace(',', ''))
        })

neighbor_states_df = pd.DataFrame(all_years_records)
print(f"Total rows: {len(neighbor_states_df)}")
neighbor_states_df.sort_values(['state', 'year'])

Total rows: 15


,year,state,city,tax_rate_pct,tax_rate_rank,tax_bill,tax_bill_rank,median_home_value
0,2023,Colorado,Denver,0.539,49,3232.0,28,599500.0
5,2024,Colorado,Denver,0.519,49,3253.0,30,626500.0
10,2025,Colorado,Denver,0.485,51,3085.0,36,636400.0
1,2023,Kansas,Wichita,1.125,32,2112.0,48,187800.0
6,2024,Kansas,Wichita,1.107,31,2193.0,48,198100.0
11,2025,Kansas,Wichita,1.118,29,2381.0,46,212900.0
2,2023,Missouri,Kansas City,1.340,17,3207.0,29,239400.0
7,2024,Missouri,Kansas City,1.369,15,3330.0,26,243200.0
12,2025,Missouri,Kansas City,1.594,11,4169.0,23,261600.0
3,2023,Nebraska,Omaha,1.983,9,4835.0,14,243800.0


## Validation

15 rows (5 states × 3 years, 2023–2025). No unmatched rows across any year.
Spot-checked 2025 Kansas ($2,381, $212,900 home) and 2025 Colorado ($3,085,
$636,400 home) against source PDF.

In [9]:
conn = sqlite3.connect("../data/processed/kansas_tax.db")

neighbor_states_df.to_sql("neighbor_state_comparison", conn, if_exists="replace", index=False)
neighbor_states_df.to_csv("../data/processed/neighbor_state_comparison.csv", index=False)

check = pd.read_sql("SELECT * FROM neighbor_state_comparison ORDER BY state, year", conn)
print(f"Rows in table: {len(check)}")

Rows in table: 15


## Early finding

Wichita (KS) has the lowest median tax bill of all five comparison cities
in every year 2023–2025 ($2,381 in 2025, vs. $3,085 Denver, $3,383 Oklahoma
City, $4,169 Kansas City, $4,857 Omaha) — driven partly by Wichita's lower
median home value ($212,900) relative to the other cities.

Notable: Colorado's effective tax rate on Denver fell from 0.539% (2023) to
0.485% (2025) even as Denver's median home value rose from $599,500 to
$636,400 — consistent with Colorado's known assessment-limit policy actively
suppressing tax growth despite rising valuations, in contrast to Kansas.

Caveat: this compares each state's largest city, not a statewide average —
Wichita's low bill partly reflects Wichita's lower home values, not
necessarily a lower burden relative to income or a Kansas-wide pattern.